In [1]:
import numpy as np, math
from scipy.integrate import solve_ivp
from scipy.optimize import minimize_scalar

import matplotlib.pyplot as plt
from scipy.optimize import brentq

In [2]:
# Constants
G=6.674e-11 
mE=5.972e24
mM=7.348e22
d=3.844e8

In [ ]:
# System paramters and basic motion


w =math.sqrt(G*(mE+mM)/d**3) # angular velocity of the system
T = 2*math.pi/w # period of the system

rE= mM*d/(mE+mM) # distance of earth from barycentre
rM= mE*d/(mE+mM) # distance of moon from barycentre
dr_approx = d*(mM/(3*mE))**(1/3) 


def body_pos(t): 
    cos_comp ,sine_comp = np.cos(w*t), np.sin(w*t)
    posE = np.array([-rE*cos_comp, -rE*sin_comp])
    posM = np.array([rM*cos_comp, rM*sin_comp])
    return posE, posM

def body_vel(t):                                           #call using 

    cos_comp, sin_comp = np.cos(w*t), np.sin(w*t)

    velE = np.array([rE*w*sin_comp, -rE*w*cos_comp])
    velM = np.array([-rM*w*sin_comp, rM*w*cos_comp])

    return velE, velM

def initial_conditions_planets(t0):             # call using (posE, posM), (velE, velM) = initial_conditions_planets(t0)

    posE, posM = body_pos(t0)
    velE, velM = body_vel(t0)

    return (posE, posM), (velE, velM)

def init(dr, t):                                                  # call using posS, velS = init(dr, t0)

    cos_comp, sin_comp = np.cos(w*t), np.sin(w*t)

    pos = np.array([(rM+dr)*cos_comp,(rM+dr)*sin_comp])
    vel = np.array([-w*(rM+dr)*sin_comp,w*(rM+dr)*cos_comp])

    return pos, vel

def func_to_solve(t, s):

    pos = s[:2]
    vel = s[2:]
    posE, posM = body_pos(t)
    dE = pos - posE
    dM = pos - posM
    dE3 = np.linalg.norm(dE)**3
    dM3 = np.linalg.norm(dM)**3
    acc = -G*mE*dE/dE3 - G*mM*dM/dM3

    return np.concatenate((vel, acc))


In [ ]:
#Scipy Solver 
def Scipy_solver(func,t_span, s, t_eval):
    posS, velS = init(dr, T0)
    s = np.concatenate((posS, velS))
    sol = solve_ivp(func, t_span, s , t_eval=t_eval, method='RK45', rtol=1e-9, atol=1e-9)
    return sol 

#Scipy Solver with dense output - use this one as you can interpolate better
def Scipy_solver_dense(func, T0, T, n, dr):
    pos,vel = init(dr, T0)
    s=np.concatenate((pos,vel))
    t = np.linspace(T0, T, n)
    sol = solve_ivp(func, [T0, T], s,t_eval=t,method='RK45',rtol=1e-9,atol=1e-9,dense_output=True)
    positions = sol.y[:2].T
    velocities = sol.y[2:].T
    times = sol.t

    return positions, velocities, times 


In [5]:
#Taylor Solver - add jerk 
def Taylor_solver(dr , T0, T,h=10.0 , store = 200):
    
    N=int(round((T-T0)/h))
    h=(T-T0)/N

    pos ,vel = init(dr,T0)    
    out_pos = []
    out_vel = []
    times = np.array([T0])

    for i in  range(N+1):
        t=T0+i*h
        
        
        if i % store ==0 or i == N:
            out_pos.append()
            out_vel.append()
            times.append(t)

        if i == N:
            break
        
        deriv=func_to_solve(t, np.concatenate((pos,vel)))
        acc = deriv[2:]
        new_pos = pos + h*vel + 0.5*h**2*acc
        new_vel = vel + h*acc

        pos, vel = new_pos, new_vel

    return np.array(out_pos), np.array(out_vel), np.array(times)
